# 02 — PhoBERT Sentiment Fine-Tuning

**Environment**: Kaggle (GPU T4 / P100). Runs the full sentiment training and inference pipeline via DVC.

**Prerequisites**: Notebook 01 must have run and pushed its outputs.

**Pipeline stages executed** (in order):
1. Pull preprocess outputs (`pipelines/preprocess`)
2. `prepare_training_data` → `merge_annotations` → `train_classifier`
3. `infer_cafef` → `merge_articles_sentiment` → `validate_*` → `build_model_frame`

**Outputs pushed to DVC remote**:
- `models/phobert-sentiment/latest/` (fine-tuned model)
- `data/sentiment/article_sentiment_scores.parquet`
- `data/interim/modeling_ready.parquet`

> **Note**: `bootstrap_labels` (LLM annotation) is a one-time manual step.
> `data/interim/training_annotations.csv` must already be committed/tracked in DVC.

In [ ]:
# ── 1. Environment Setup ────────────────────────────────────────────────────
import os, subprocess, sys
from pathlib import Path

REPO_URL    = "https://github.com/tlong-ds/news-sentiment-analysis.git"
REPO_BRANCH = "main"
REPO_DIR    = Path("/kaggle/working/news-sentiment-analysis")
ON_KAGGLE   = Path("/kaggle").exists()

def run(cmd: list[str], cwd: Path = REPO_DIR) -> None:
    subprocess.run(cmd, cwd=cwd, check=True)
    print(f"✓ {' '.join(str(c) for c in cmd)}")

if ON_KAGGLE:
    if not REPO_DIR.exists():
        run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1",
             REPO_URL, str(REPO_DIR)], cwd=Path("/kaggle/working"))
    run([sys.executable, "-m", "pip", "install", "-q", "-r",
         str(REPO_DIR / "requirements.txt")])
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
else:
    REPO_DIR = Path().resolve()

PREPROCESS_PIPELINE = REPO_DIR / "pipelines" / "preprocess"
SENTIMENT_PIPELINE  = REPO_DIR / "pipelines" / "sentiment"
print(f"ROOT: {REPO_DIR}")

In [ ]:
# ── 2. DVC Remote Configuration ─────────────────────────────────────────────
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient  # noqa: import
    sa_json = UserSecretsClient().get_secret("GDRIVE_SERVICE_ACCOUNT_JSON")
    sa_path = REPO_DIR / ".dvc" / "gdrive_sa.json"
    sa_path.write_text(sa_json)
    run(["dvc", "remote", "modify", "gdrive",
         "gdrive_use_service_account", "true"])
    run(["dvc", "remote", "modify", "gdrive",
         "gdrive_service_account_json_file_path", str(sa_path)])
    print("DVC remote configured.")

In [ ]:
# ── 3. Pull Preprocess Outputs + Annotation File ────────────────────────────
# Pull the outputs produced by notebook 01 (preprocess pipeline)
run(["dvc", "pull", "--run-cache", "-q"], cwd=PREPROCESS_PIPELINE)
# Pull the committed annotation CSV (produced by bootstrap_labels, committed once)
run(["dvc", "pull", "data/interim/training_annotations.csv", "-q"])

In [ ]:
# ── 4. Training Pipeline ─────────────────────────────────────────────────────
# Stages: prepare_training_data → merge_annotations → train_classifier
run(["dvc", "repro",
     "prepare_training_data",
     "merge_annotations",
     "train_classifier"],
    cwd=SENTIMENT_PIPELINE)

In [ ]:
# ── 5. Inference + Validation + Model Frame ──────────────────────────────────
# Stages: infer_cafef → merge_articles_sentiment
#         → validate_inference → validate_daily_aggregation → build_model_frame
run(["dvc", "repro",
     "infer_cafef",
     "merge_articles_sentiment",
     "validate_inference",
     "validate_daily_aggregation",
     "build_model_frame"],
    cwd=SENTIMENT_PIPELINE)

In [ ]:
# ── 6. Push All Outputs ──────────────────────────────────────────────────────
run(["dvc", "push"], cwd=SENTIMENT_PIPELINE)
print("All outputs pushed to DVC remote.")

In [ ]:
# ── 7. Verify ────────────────────────────────────────────────────────────────
import json, yaml, pandas as pd

params   = yaml.safe_load((REPO_DIR / "params.yaml").read_text())
interim  = REPO_DIR / params["paths"]["interim_dir"]
sent_dir = REPO_DIR / params["paths"]["sentiment_dir"]
model_dir = REPO_DIR / params["paths"]["model_dir"]

train_report = json.loads((model_dir / "training_report.json").read_text())
eval_report  = json.loads((model_dir / "evaluation.json").read_text())
print("=== Classifier Training Report ===")
print(f"  Rows         : {train_report['rows']:,}")
print(f"  Splits       : {train_report['splits']}")
print(f"  Test accuracy: {eval_report['test']['accuracy']:.4f}")
print(f"  Label dist   : {eval_report['test']['label_distribution']}")

scores = pd.read_parquet(sent_dir / "article_sentiment_scores.parquet")
print(f"\nSentiment scores : {len(scores):,} articles")
print(scores["sentiment_label"].value_counts().to_string())

mf = pd.read_parquet(interim / "modeling_ready.parquet")
print(f"\nModeling frame   : {len(mf):,} rows | {mf['date'].min()} → {mf['date'].max()}")